In [3]:
import pandas as pd
import seaborn
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

In [4]:
lang_compare = pd.read_csv("among_lang_metrics.csv", index_col=0)
baseline_compare = pd.read_csv("between_lang_baseline_metrics.csv", index_col=0)

In [5]:
lang_compare.head() # compare between languages eg french1<->english1, french2<->english2, french3<->english3 ... 
                    # and between baselines eg baseline_french1<->baseline_english1 ...

,Model,Question,Language,Cosine,Rouge
0,Qwen3-32B,Question 1,NL,0.9173,0.2600
1,Qwen3-32B,Question 1,NL,0.9105,0.2626
2,Qwen3-32B,Question 1,NL,0.9320,0.2591
3,Qwen3-32B,Question 1,NL,0.9314,0.2604
4,Qwen3-32B,Question 1,NL,0.8965,0.2653


In [6]:
baseline_compare.head() # compare specific language and its baseline eg french1<->baseline_french1, french2<->baseline_french2 ...

,Model,Question,Cosine,fBaseline,Language1-Language2
0,Qwen3-32B,Question 1,0.9358,False,NL-EN
1,Qwen3-32B,Question 1,0.9361,False,NL-EN
2,Qwen3-32B,Question 1,0.9462,False,NL-EN
3,Qwen3-32B,Question 1,0.9462,False,NL-EN
4,Qwen3-32B,Question 1,0.9453,False,NL-EN


In [9]:
lang_compare.drop("Question", axis=1).groupby(["Model","Language"]).agg(['mean', 'std', 'count'])

Cosine                     Rouge                
                        mean       std count      mean       std count
Model     Language                                                    
Qwen3-32B EN        0.923821  0.043929  1100  0.268407  0.074112  1100
          FR        0.878665  0.031775  1100  0.251704  0.044027  1100
          NL        0.852181  0.066231  1100  0.240205  0.063306  1100

In [ ]:
lang_compare.groupby(["Model","Question","Language"]).agg(['mean', 'std', 'count'])

In [ ]:
baseline_compare.drop("Question", axis=1).groupby(["Model", "Language"]).agg(["mean", "count"])

In [ ]:
baseline_compare.groupby(["Model", "Question", "Language"]).agg(["mean", "count"])

In [ ]:
seaborn.set(style = 'whitegrid')  
ys = ["Cosine", "Rouge"]
fig, axs = plt.subplots(ncols=len(ys), figsize=(15,5), sharey=True)

# not between questions, each violin plot has 24 data points
for i in range(len(ys)):
    axs[i].tick_params(labelrotation=45)   
    axs[i].title.set_text(ys[i])
    seaborn.violinplot(x="Model", 
                       y=ys[i], 
                       hue="Language",
                       data=baseline_compare,
                       density_norm='count',
                       ax=axs[i])

In [ ]:
seaborn.set(style = 'whitegrid')  
ys = ["Cosine", "Rouge"]

# between questions, each violin plot only has 3 data points
for y in ys:
    fig, axs = plt.subplots(ncols=len(baseline_compare["Model"].unique()), figsize=(20,5),  sharey=True)
    fig.suptitle(y)
    i = 0
    for model in baseline_compare["Model"].unique():  
        axs[i].title.set_text(model)
        axs[i].tick_params(labelrotation=45)                    
        seaborn.violinplot(x ="Question", 
                           y=y, 
                           hue="Language",
                           data = baseline_compare[baseline_compare["Model"]==model],
                           density_norm='count',
                           ax=axs[i])
        i+=1
    plt.figure()

In [ ]:
seaborn.set(style = 'whitegrid')  
ys = ["Cosine", "Rouge"]

# between questions, each violin plot only has 3 data points
for y in ys:
    fig, axs = plt.subplots(ncols=len(lang_compare["Model"].unique())+2, figsize=(30,5),  sharey=True)
    fig.suptitle(y)
    i = 0
    for model in lang_compare["Model"].unique():  
        for b in [True, False]:
            axs[i].title.set_text(f"{model}_baseline{b}")
            axs[i].tick_params(labelrotation=45)                    
            seaborn.violinplot(x ="Question", 
                               y=y, 
                               hue="Language1-Language2",
                               data = lang_compare[(lang_compare["Model"]==model)&(lang_compare["fBaseline"]==b)],
                               density_norm='count',
                               ax=axs[i])
            i+=1
    plt.figure()

In [ ]:
seaborn.set(style = 'whitegrid')  
ys = ["Cosine", "Rouge"]

# not between questions, each violin plot has 24 data points
for y in ys:
    fig, axs = plt.subplots(ncols=len(lang_compare["Model"].unique()), figsize=(20,5),  sharey=True)
    fig.suptitle(y)
    i = 0
    for model in lang_compare["Model"].unique():  
        axs[i].title.set_text(f"{model}")
        axs[i].tick_params(labelrotation=45)                    
        seaborn.violinplot(x ="Language1-Language2", 
                           y=y, 
                           hue="fBaseline",
                           data = lang_compare[(lang_compare["Model"]==model)],
                           density_norm='count',
                           ax=axs[i])
        i+=1
    plt.figure()

## See data

In [ ]:
from datasets import load_from_disk, concatenate_datasets
import pickle
import json
from glob import glob

paths = glob("t0*")

In [ ]:
answers = [load_from_disk(x) for x in paths]
answers = concatenate_datasets(answers)
with open("question_map.pkl", 'rb') as f:
    question_map = pickle.load(f)
with open("params.json", 'r') as f:
    params = json.load(f)
answers = answers.add_column("question_mapped", [question_map[x] for x in answers["question"]])

In [ ]:
question_map

In [ ]:
answers = answers.to_pandas()

In [ ]:
answers["answers"][1][0]

In [ ]:
answers.to_csv("all_answers.csv", encoding="utf-8")